# 39 -- Section 01e Step 2: Rumble term + other 529 movies, both scenarios

Reuses notebook 37/38's exact user, movie, seeds and starting weights. Splits
x+ and x- into Rumble's own term and the sum over the other 529 rated movies,
directly for each of the 01c and 01e scenarios (not via the b * sum-of-weights
shortcut), so the worked-example table can show one consistent breakdown per
column. Asserts every result reproduces the page's current x values before
any number is written. Does not touch the HTML.

## Part 0 -- Setup: scan notebooks/outputs, pick this notebook's number

In [1]:
from pathlib import Path
import json
import re

import pandas as pd
import torch

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
notebooks_dir = root / "notebooks"
data_dir = root / "data"
out_dir = root / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

nb_paths = sorted(p for p in notebooks_dir.glob("*.ipynb") if "table_merge_breakdown" not in p.stem)
num_re = re.compile(r"^(\d+)")
numbered = sorted(
    (int(m.group(1)), p) for p in nb_paths if (m := num_re.match(p.stem))
)
highest_num, highest_path = numbered[-1]
this_num = highest_num + 1
this_name = f"{this_num}_step2_table_merge_breakdown.ipynb"
print(f"highest existing notebook: {highest_num:>3d}  ({highest_path.name})")
print(f"this notebook is:          {this_num:>3d}  ({this_name})")

highest existing notebook:  38  (38_step2_rumble_term_breakdown.ipynb)
this notebook is:           39  (39_step2_table_merge_breakdown.ipynb)


## Part 1 -- Reuse notebook 37/38's exact setup

In [2]:
train_path = data_dir / "train_hyperbolic.csv"
test_path = data_dir / "test_hyperbolic.csv"
results31_path = out_dir / "31_step8_results.json"
results37_path = out_dir / "37_step1to6_rating_habit_b_results.json"
for p in (train_path, test_path, results31_path, results37_path):
    assert p.exists(), f"Missing {p}"

train_hyp = pd.read_csv(train_path)
test_hyp = pd.read_csv(test_path)
assert len(train_hyp) == 492055 and len(test_hyp) == 123474

results31 = json.loads(results31_path.read_text())
results37 = json.loads(results37_path.read_text())
REAL_SEED = results31["seeds"]["plus_seed_reused"]
MINUS_SEED = results31["seeds"]["minus_seed_trained"]
N_HIDDEN = int(results31["architecture_match"]["128 hidden units"].split("=")[1])
H1_IDX = 0

all_users = sorted(pd.concat([train_hyp["userId"], test_hyp["userId"]]).unique())
all_movies = sorted(pd.concat([train_hyp["movieId"], test_hyp["movieId"]]).unique())
n_movies = len(all_movies)
movie_to_col = {m: j for j, m in enumerate(all_movies)}

movies = pd.read_csv(data_dir / "movie.csv")
RUMBLE_ID = int(movies[movies["title"].str.contains("Rumble in the Bronx", case=False, na=False)].iloc[0]["movieId"])
rumble_col = movie_to_col[RUMBLE_ID]
MEDIAN_USER = results31["worked_example_plus"]["user"]

user_rows = train_hyp[train_hyp["userId"] == MEDIAN_USER]
v0_full = torch.zeros(n_movies)
mask0_full = torch.zeros(n_movies)
for row in user_rows.itertuples():
    v0_full[movie_to_col[row.movieId]] = row.a
    mask0_full[movie_to_col[row.movieId]] = 1.0
rated_idx = mask0_full.nonzero(as_tuple=True)[0]
a_rumble = v0_full[rumble_col].item()
n_rated = len(rated_idx)
assert n_rated == results37["step2"]["R_size"] == 530

g_init_p = torch.Generator().manual_seed(REAL_SEED)
W_plus_init = torch.randn(n_movies, N_HIDDEN, generator=g_init_p) * 0.01
g_init_m = torch.Generator().manual_seed(MINUS_SEED)
W_minus_init = torch.randn(n_movies, N_HIDDEN, generator=g_init_m) * 0.01

b_val = results37["step1"]["b"]
w_plus_rumble = W_plus_init[rumble_col, H1_IDX].item()
w_minus_rumble = W_minus_init[rumble_col, H1_IDX].item()
print(f"a={a_rumble:.6f}  b={b_val:.6f}  n_rated={n_rated}")
print(f"w+[Rumble,h1]={w_plus_rumble:.6f}  w-[Rumble,h1]={w_minus_rumble:.6f}")

a=0.777778  b=0.656813  n_rated=530
w+[Rumble,h1]=0.006433  w-[Rumble,h1]=-0.009362


## Part 2 -- x+/x- for both scenarios, split into Rumble term and the other 529 movies

In [3]:
v_plus_01e_full = torch.where(mask0_full.bool(), v0_full + b_val, torch.zeros(n_movies))
v_minus_01e_full = torch.where(mask0_full.bool(), v0_full - b_val, torch.zeros(n_movies))

x_plus_01c = (v0_full @ W_plus_init)[H1_IDX].item()
x_minus_01c = (v0_full @ W_minus_init)[H1_IDX].item()
x_plus_01e = (v_plus_01e_full @ W_plus_init)[H1_IDX].item()
x_minus_01e = (v_minus_01e_full @ W_minus_init)[H1_IDX].item()

assert abs(x_plus_01c - results37["step2"]["x_plus"]["01c"]) < 1e-6
assert abs(x_minus_01c - results37["step2"]["x_minus"]["01c"]) < 1e-6
assert abs(x_plus_01e - results37["step2"]["x_plus"]["01e"]) < 1e-6
assert abs(x_minus_01e - results37["step2"]["x_minus"]["01e"]) < 1e-6
print("all four x values match the page's current table (notebook 37). OK.")

channel_plus_rumble_01c = a_rumble
channel_minus_rumble_01c = a_rumble
channel_plus_rumble_01e = a_rumble + b_val
channel_minus_rumble_01e = a_rumble - b_val

rumble_term_plus_01c = channel_plus_rumble_01c * w_plus_rumble
rumble_term_minus_01c = channel_minus_rumble_01c * w_minus_rumble
rumble_term_plus_01e = channel_plus_rumble_01e * w_plus_rumble
rumble_term_minus_01e = channel_minus_rumble_01e * w_minus_rumble

rest_plus_01c = x_plus_01c - rumble_term_plus_01c
rest_minus_01c = x_minus_01c - rumble_term_minus_01c
rest_plus_01e = x_plus_01e - rumble_term_plus_01e
rest_minus_01e = x_minus_01e - rumble_term_minus_01e

for rumble_term, rest, x in (
    (rumble_term_plus_01c, rest_plus_01c, x_plus_01c),
    (rumble_term_minus_01c, rest_minus_01c, x_minus_01c),
    (rumble_term_plus_01e, rest_plus_01e, x_plus_01e),
    (rumble_term_minus_01e, rest_minus_01e, x_minus_01e),
):
    assert abs((rumble_term + rest) - x) < 1e-9
print("Rumble term + other 529 movies reproduces x in all four cases, to 1e-9. OK.")

print(f"01c: rumble+={rumble_term_plus_01c:.6f}  rest+={rest_plus_01c:.6f}  x+={x_plus_01c:.6f}")
print(f"01c: rumble-={rumble_term_minus_01c:.6f}  rest-={rest_minus_01c:.6f}  x-={x_minus_01c:.6f}")
print(f"01e: rumble+={rumble_term_plus_01e:.6f}  rest+={rest_plus_01e:.6f}  x+={x_plus_01e:.6f}")
print(f"01e: rumble-={rumble_term_minus_01e:.6f}  rest-={rest_minus_01e:.6f}  x-={x_minus_01e:.6f}")

all four x values match the page's current table (notebook 37). OK.
Rumble term + other 529 movies reproduces x in all four cases, to 1e-9. OK.
01c: rumble+=0.005004  rest+=-0.216119  x+=-0.211115
01c: rumble-=-0.007281  rest-=-0.198086  x-=-0.205367
01e: rumble+=0.009229  rest+=-0.438629  x+=-0.429400
01e: rumble-=-0.001132  rest-=0.009477  x-=0.008345


## Part 3 -- Save for the rebuilt Step 2 worked-example table

In [4]:
results = {
    "notebook": this_name,
    "reused_from": "31_step8_minus_channel.ipynb, 37_step1to6_rating_habit_b.ipynb, 38_step2_rumble_term_breakdown.ipynb",
    "movie": "Rumble in the Bronx (Hont faan kui) (1995)",
    "user": MEDIAN_USER,
    "hidden_unit": "h1 (index 0)",
    "a": a_rumble,
    "b": b_val,
    "n_rated": n_rated,
    "w_plus_rumble": w_plus_rumble,
    "w_minus_rumble": w_minus_rumble,
    "channel_plus_rumble": {"01c": channel_plus_rumble_01c, "01e": channel_plus_rumble_01e},
    "channel_minus_rumble": {"01c": channel_minus_rumble_01c, "01e": channel_minus_rumble_01e},
    "rumble_term_plus": {"01c": rumble_term_plus_01c, "01e": rumble_term_plus_01e},
    "rumble_term_minus": {"01c": rumble_term_minus_01c, "01e": rumble_term_minus_01e},
    "rest_plus": {"01c": rest_plus_01c, "01e": rest_plus_01e},
    "rest_minus": {"01c": rest_minus_01c, "01e": rest_minus_01e},
    "x_plus": {"01c": x_plus_01c, "01e": x_plus_01e},
    "x_minus": {"01c": x_minus_01c, "01e": x_minus_01e},
}
results_path = out_dir / "39_step2_table_merge_breakdown_results.json"
with results_path.open("w") as f:
    json.dump(results, f, indent=2)
print(f"saved {results_path}")
print(json.dumps(results, indent=2))

saved /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/outputs/39_step2_table_merge_breakdown_results.json
{
  "notebook": "39_step2_table_merge_breakdown.ipynb",
  "reused_from": "31_step8_minus_channel.ipynb, 37_step1to6_rating_habit_b.ipynb, 38_step2_rumble_term_breakdown.ipynb",
  "movie": "Rumble in the Bronx (Hont faan kui) (1995)",
  "user": 24127,
  "hidden_unit": "h1 (index 0)",
  "a": 0.7777777910232544,
  "b": 0.6568134171907757,
  "n_rated": 530,
  "w_plus_rumble": 0.006433217320591211,
  "w_minus_rumble": -0.009361893869936466,
  "channel_plus_rumble": {
    "01c": 0.7777777910232544,
    "01e": 1.4345912082140302
  },
  "channel_minus_rumble": {
    "01c": 0.7777777910232544,
    "01e": 0.12096437383247871
  },
  "rumble_term_plus": {
    "01c": 0.005003613556781972,
    "01e": 0.009229037008650371
  },
  "rumble_term_minus": {
    "01c": -0.007281473133953331,
    "01e": -0.0011324556298629854
  },
  "rest_plus": {
    "01c": -0.21611873539821264,
    "01e": -0.4